In [1]:
import pandas as pd
import numpy as np
import time
import pickle
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

print('Loading flood dataset...')
df = pd.read_csv('../data/floods/floods_processed.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nTarget distribution:')
print(df['severity'].value_counts())

Loading flood dataset...
Shape: (40796, 9)
Columns: ['BEGIN_LAT', 'BEGIN_LON', 'damage_numeric', 'deaths_total', 'severity', 'EVENT_TYPE', 'STATE', 'EVENT_TYPE_CODE', 'STATE_CODE']

Target distribution:
severity
0    35789
1     4159
2      848
Name: count, dtype: int64


In [1]:
print("""
=== DATA LEAKAGE IDENTIFIED AND FIXED ===

Initial run with damage_numeric and deaths_total as features
returned F1 = 1.0 — suspiciously perfect accuracy.

Root cause: These values only exist AFTER a flood occurs.
A real prediction system would never have them at prediction time.

Fix: Remove leaky features. Retrain using only:
  - Location (BEGIN_LAT, BEGIN_LON)
  - Flood type (EVENT_TYPE_CODE)  
  - State (STATE_CODE)

Result after fix: F1 = 0.8445 — genuinely useful prediction.
""")


=== DATA LEAKAGE IDENTIFIED AND FIXED ===

Initial run with damage_numeric and deaths_total as features
returned F1 = 1.0 — suspiciously perfect accuracy.

Root cause: These values only exist AFTER a flood occurs.
A real prediction system would never have them at prediction time.

Fix: Remove leaky features. Retrain using only:
  - Location (BEGIN_LAT, BEGIN_LON)
  - Flood type (EVENT_TYPE_CODE)  
  - State (STATE_CODE)

Result after fix: F1 = 0.8445 — genuinely useful prediction.



In [3]:
# Remove leaky features - damage and deaths happen AFTER the flood
# Model should predict risk BEFORE using only location and time
feature_cols = ['BEGIN_LAT', 'BEGIN_LON', 'EVENT_TYPE_CODE', 'STATE_CODE']

X = df[feature_cols].fillna(0)
y = df['severity']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

results = {}

print('Training Random Forest...')
start = time.time()
rf = RandomForestClassifier(n_estimators=100, random_state=42,
                             n_jobs=-1, class_weight='balanced')
rf.fit(X_train, y_train)
rf_f1 = f1_score(y_test, rf.predict(X_test), average='weighted')
results['Random Forest'] = {'f1': rf_f1, 'time': time.time()-start}
print(f'  F1: {rf_f1:.4f} | Time: {results["Random Forest"]["time"]:.1f}s')

print('Training XGBoost...')
start = time.time()
xgb = XGBClassifier(n_estimators=100, random_state=42,
                     n_jobs=-1, verbosity=0)
xgb.fit(X_train, y_train)
xgb_f1 = f1_score(y_test, xgb.predict(X_test), average='weighted')
results['XGBoost'] = {'f1': xgb_f1, 'time': time.time()-start}
print(f'  F1: {xgb_f1:.4f} | Time: {results["XGBoost"]["time"]:.1f}s')

print('Training LightGBM...')
start = time.time()
lgbm = LGBMClassifier(n_estimators=100, random_state=42,
                       n_jobs=-1, verbose=-1)
lgbm.fit(X_train, y_train)
lgbm_f1 = f1_score(y_test, lgbm.predict(X_test), average='weighted')
results['LightGBM'] = {'f1': lgbm_f1, 'time': time.time()-start}
print(f'  F1: {lgbm_f1:.4f} | Time: {results["LightGBM"]["time"]:.1f}s')

print('Training CatBoost...')
start = time.time()
cat = CatBoostClassifier(iterations=100, random_seed=42,
                          verbose=0, auto_class_weights='Balanced')
cat.fit(X_train, y_train)
cat_f1 = f1_score(y_test, cat.predict(X_test), average='weighted')
results['CatBoost'] = {'f1': cat_f1, 'time': time.time()-start}
print(f'  F1: {cat_f1:.4f} | Time: {results["CatBoost"]["time"]:.1f}s')

print('\n' + '='*40)
print(f'{"MODEL":<20} {"F1 SCORE":>10} {"TIME":>8}')
print('='*40)
winner_name = max(results, key=lambda x: results[x]['f1'])
for model, r in results.items():
    marker = ' <-- winner' if model == winner_name else ''
    print(f'{model:<20} {r["f1"]:>10.4f} {r["time"]:>7.1f}s{marker}')
print('='*40)
print('\nNote: Features are location + flood type only')
print('Damage and deaths excluded to prevent data leakage')

Training Random Forest...
  F1: 0.8445 | Time: 0.5s
Training XGBoost...
  F1: 0.8429 | Time: 0.3s
Training LightGBM...
  F1: 0.8382 | Time: 1.0s
Training CatBoost...
  F1: 0.6524 | Time: 0.3s

MODEL                  F1 SCORE     TIME
Random Forest            0.8445     0.5s <-- winner
XGBoost                  0.8429     0.3s
LightGBM                 0.8382     1.0s
CatBoost                 0.6524     0.3s

Note: Features are location + flood type only
Damage and deaths excluded to prevent data leakage


In [4]:
import joblib
import os
os.makedirs('../models', exist_ok=True)
joblib.dump(rf, '../models/rf_flood.joblib', compress=3)
print('Flood RF saved')

Flood RF saved
